In [2]:
%pip install numpy

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.2.4-cp310-cp310-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.2.4-cp310-cp310-win_amd64.whl (12.9 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.0 requires numpy<2.2,>=1.24, but you have numpy 2.2.4 which is incompatible.
tensorflow 2.18.1 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.4 which is incompatible.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.1 which is incompatible.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.4 which is incompatible.
ultralytics 8.3.91 requires numpy<=2.1.1,>=1.23.0, but you have numpy 2.2.4 which is incompatible.


### Data Preprocessing & Exploration

#### Step 1: Load the Datasets

In [9]:
import pandas as pd

# Load the datasets
demand_data = pd.read_csv('dataset\demand_forecasting.csv')
inventory_data = pd.read_csv('dataset\inventory_monitoring.csv')
pricing_data = pd.read_csv('dataset\pricing_optimization.csv')

# View the first few rows of each dataset
print("Demand Forecasting Data:")
print(demand_data.head())
print("\n")
print("Inventory Monitoring Data:")
print(inventory_data.head())
print("\n")
print("Pricing Optimization Data:")
print(pricing_data.head())

Demand Forecasting Data:
   Product ID        Date  Store ID  Sales Quantity  Price Promotions  \
0        4277  2024-01-03        48             330  24.38         No   
1        5540  2024-04-29        10             334  74.98        Yes   
2        5406  2024-01-11        67             429  24.83        Yes   
3        5617  2024-04-04        17             298  13.41         No   
4        3480  2024-12-14        33             344  94.96        Yes   

  Seasonality Factors    External Factors Demand Trend Customer Segments  
0            Festival  Competitor Pricing   Increasing           Regular  
1             Holiday             Weather       Stable           Premium  
2             Holiday  Economic Indicator   Decreasing           Premium  
3                 NaN  Economic Indicator       Stable           Regular  
4            Festival             Weather   Increasing           Regular  


Inventory Monitoring Data:
   Product ID  Store ID  Stock Levels  Supplier Lead Time

In [10]:
print("Demand Forecasting columns:")
print(demand_data.columns)
print("\n")
print("Inventory Monitoring columns:")
print(inventory_data.columns)
print("\n")
print("Pricing Optimization columns:")
print(pricing_data.columns)

Demand Forecasting columns:
Index(['Product ID', 'Date', 'Store ID', 'Sales Quantity', 'Price',
       'Promotions', 'Seasonality Factors', 'External Factors', 'Demand Trend',
       'Customer Segments'],
      dtype='object')


Inventory Monitoring columns:
Index(['Product ID', 'Store ID', 'Stock Levels', 'Supplier Lead Time (days)',
       'Stockout Frequency', 'Reorder Point', 'Expiry Date',
       'Warehouse Capacity', 'Order Fulfillment Time (days)'],
      dtype='object')


Pricing Optimization columns:
Index(['Product ID', 'Store ID', 'Price', 'Competitor Prices', 'Discounts',
       'Sales Volume', 'Customer Reviews', 'Return Rate (%)', 'Storage Cost',
       'Elasticity Index'],
      dtype='object')


#### Step 2: Data Cleaning

In [12]:
# Check for missing values
print("Missing values in Demand Forecasting:")
print(demand_data.isnull().sum())
print("\n")
print("Missing values in Inventory Monitoring:")
print(inventory_data.isnull().sum())
print("\n")
print("Missing values in Pricing Optimization:")
print(pricing_data.isnull().sum())


Missing values in Demand Forecasting:
Product ID             0
Date                   0
Store ID               0
Sales Quantity         0
Price                  0
Promotions             0
Seasonality Factors    0
External Factors       0
Demand Trend           0
Customer Segments      0
dtype: int64


Missing values in Inventory Monitoring:
Product ID                       0
Store ID                         0
Stock Levels                     0
Supplier Lead Time (days)        0
Stockout Frequency               0
Reorder Point                    0
Expiry Date                      0
Warehouse Capacity               0
Order Fulfillment Time (days)    0
dtype: int64


Missing values in Pricing Optimization:
Product ID           0
Store ID             0
Price                0
Competitor Prices    0
Discounts            0
Sales Volume         0
Customer Reviews     0
Return Rate (%)      0
Storage Cost         0
Elasticity Index     0
dtype: int64


In [13]:
# Handle missing values
demand_data.ffill(inplace=True)
inventory_data.ffill(inplace=True) 
pricing_data.ffill(inplace=True)

# Convert date columns to datetime format
demand_data['Date'] = pd.to_datetime(demand_data['Date'])
inventory_data['Expiry Date'] = pd.to_datetime(inventory_data['Expiry Date'])

# Check for duplicates
demand_data.drop_duplicates(inplace=True)
inventory_data.drop_duplicates(inplace=True)
pricing_data.drop_duplicates(inplace=True)

#### Step 3: Feature Engineering (Optional)

In [14]:
# Add month and day of the week columns
demand_data['Month'] = demand_data['Date'].dt.month
demand_data['DayOfWeek'] = demand_data['Date'].dt.dayofweek

# Example of creating a seasonal feature
demand_data['SeasonalIndex'] = demand_data['Seasonality Factors'].apply(lambda x: 1 if 'Holiday' in x else 0)

### Building AI Models

#### Step 1: Demand Forecasting Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Select features and target
X_demand = demand_data[['Price', 'Promotions', 'SeasonalIndex', 'External Factors', 'Customer Segments']]
y_demand = demand_data['Sales Quantity']

# One-hot encode categorical features
X_demand = pd.get_dummies(X_demand)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_demand, y_demand, test_size=0.2, random_state=42)

# Initialize and train the model
model_demand = RandomForestRegressor(n_estimators=100, random_state=42)
model_demand.fit(X_train, y_train)

# Predict and evaluate the model
y_pred_demand = model_demand.predict(X_test)
mse_demand = mean_squared_error(y_test, y_pred_demand)
print(f'Mean Squared Error for Demand Forecasting: {mse_demand}')


Mean Squared Error for Demand Forecasting: 25552.0073765594


#### Step 2: Stock Optimization Model

In [18]:
from sklearn.tree import DecisionTreeRegressor

# Select features and target
X_inventory = inventory_data[['Supplier Lead Time (days)', 'Stockout Frequency', 'Reorder Point']]
y_inventory = inventory_data['Stock Levels']

# Train-test split
X_train_inv, X_test_inv, y_train_inv, y_test_inv = train_test_split(X_inventory, y_inventory, test_size=0.2, random_state=42)

# Initialize and train the model
model_inventory = DecisionTreeRegressor(random_state=42)
model_inventory.fit(X_train_inv, y_train_inv)

# Predict and evaluate the model
y_pred_inventory = model_inventory.predict(X_test_inv)
mse_inventory = mean_squared_error(y_test_inv, y_pred_inventory)
print(f'Mean Squared Error for Inventory Optimization: {mse_inventory}')

Mean Squared Error for Inventory Optimization: 156740.600875


#### Step 3: Pricing Optimization Model

In [19]:
from sklearn.linear_model import ElasticNet

# Select features and target
X_pricing = pricing_data[['Competitor Prices', 'Discounts', 'Sales Volume', 'Return Rate (%)', 'Elasticity Index']]
y_pricing = pricing_data['Price']

# Train-test split
X_train_price, X_test_price, y_train_price, y_test_price = train_test_split(X_pricing, y_pricing, test_size=0.2, random_state=42)

# Initialize and train the model
model_pricing = ElasticNet(random_state=42)
model_pricing.fit(X_train_price, y_train_price)

# Predict and evaluate the model
y_pred_pricing = model_pricing.predict(X_test_price)
mse_pricing = mean_squared_error(y_test_price, y_pred_pricing)
print(f'Mean Squared Error for Pricing Optimization: {mse_pricing}')

Mean Squared Error for Pricing Optimization: 769.7947306898345


### Multi-Agent System Design

#### Step 1: Agent Class Design

In [20]:
class ForecastingAgent:
    def __init__(self, model):
        self.model = model

    def forecast_demand(self, features):
        return self.model.predict(features)

class StockOptimizationAgent:
    def __init__(self, model):
        self.model = model

    def optimize_stock(self, features):
        return self.model.predict(features)

class SupplierAgent:
    def restock(self, store_id, product_id, stock_needed):
        # Call supplier API or database to place the restocking order
        print(f'Restocking {stock_needed} units of Product {product_id} for Store {store_id}.')

class AnomalyDetectionAgent:
    def detect_anomalies(self, stock_levels, sales):
        # Identify anomalies (e.g., stockouts or excess inventory)
        if stock_levels < sales:
            print("Stockout detected!")
        elif stock_levels > 2 * sales:
            print("Excess inventory detected!")

#### Step 2: Agent Coordination

In [21]:
class RetailInventorySystem:
    def __init__(self):
        self.forecasting_agent = ForecastingAgent(model_demand)
        self.optimization_agent = StockOptimizationAgent(model_inventory)
        self.supplier_agent = SupplierAgent()
        self.anomaly_agent = AnomalyDetectionAgent()

    def manage_inventory(self, store_id, product_id, features):
        # Step 1: Forecast demand
        forecasted_demand = self.forecasting_agent.forecast_demand(features)
        
        # Step 2: Optimize stock levels
        stock_needed = self.optimization_agent.optimize_stock(features)
        
        # Step 3: Detect anomalies
        self.anomaly_agent.detect_anomalies(stock_needed, forecasted_demand)

        # Step 4: Restock if needed
        if stock_needed > forecasted_demand:
            self.supplier_agent.restock(store_id, product_id, stock_needed - forecasted_demand)

# Example of how to run the system
retail_system = RetailInventorySystem()
retail_system.manage_inventory(store_id=48, product_id=4277, features=demand_data.iloc[0][['Price', 'Promotions']])

C:\Users\DELL\AppData\Roaming\Python\Python310\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


ValueError: could not convert string to float: 'No'